In [ ]:
# %%
# ============================================================
# ☀️ SMART HMI FULL-DISK DOWNLOADER → NPZ CONVERTER (±1h→±2h)
# ============================================================
# • Downloads all 5 HMI products for each of 10 flares
# • Tries ±1h first, retries ±2h if no FITS found
# • Reads HDU[1] explicitly (safe_read_hmi)
# • Converts to 256×256 .npz under full_disk/npz_hmi/
# • Cleans up FITS after each window
# • Skips existing .npz (resumable)
# ============================================================

import os, glob, cv2, numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from astropy.io import fits
import astropy.units as u
from sunpy.net import Fido, attrs as a

# ============================================================
# 🔧 SETUP
# ============================================================

load_dotenv()
EMAIL = os.environ.get("EMAIL")
if not EMAIL:
    raise ValueError("❌ Missing EMAIL in .env — add EMAIL=your_registered_email@njit.edu")

BASE_DIR = os.path.abspath(".")
print(f"✅ Base directory: {BASE_DIR}")
print(f"✅ Using JSOC email: {EMAIL}")

# ============================================================
# ☀️ FINAL 10 FLARES
# ============================================================

FLARES = [
    ("AR11158_M6.6","2011-02-13 17:28:00"),
    ("AR11158_X2.2","2011-02-15 01:44:00"),
    ("AR11261_M9.3","2011-07-30 02:04:00"),
    ("AR11429_X5.4","2012-03-07 00:02:00"),
    ("AR11429_M6.3","2012-03-09 03:22:00"),
    ("AR11520_X1.4","2012-07-12 15:37:00"),
    ("AR11719_M6.5","2013-04-11 06:55:00"),
    ("AR12036_M7.3","2014-04-18 12:31:00"),
    ("AR11944_X1.2","2014-01-07 18:04:00"),
    ("AR12017_X1.0","2014-03-29 17:35:00"),
]

# ============================================================
# 💾 SAFE FITS READER (HDU[1])
# ============================================================

def safe_read_hmi(path):
    """Reads image data safely (HDU[1] preferred)."""
    try:
        with fits.open(path, memmap=False) as hdul:
            if len(hdul) > 1 and hdul[1].data is not None:
                return hdul[1].data.astype(np.float32)
            if hdul[0].data is not None:
                return hdul[0].data.astype(np.float32)
    except Exception as e:
        print(f"⚠️ Error reading {os.path.basename(path)}: {e}")
    return None

# ============================================================
# 💾 FITS → NPZ CONVERTER + CLEANUP
# ============================================================

def fits_to_npz_all_hmi(save_dir, out_npz, target_size=(256, 256)):
    """Combine all 5 HMI FITS → .NPZ, clean up FITS."""
    fits_files = sorted(glob.glob(os.path.join(save_dir, "*.fits")))
    if not fits_files:
        return False

    stacks = {}
    for fpath in fits_files:
        data = safe_read_hmi(fpath)
        if data is None: 
            continue
        valid = np.isfinite(data)
        if not np.any(valid): 
            continue

        lo, hi = np.percentile(data[valid], 1), np.percentile(data[valid], 99)
        data = np.clip(data, lo, hi)
        data = np.log1p(data - lo + 1e-6)
        data = cv2.resize(data, target_size)

        ch = "unknown"
        if "hmi.b_720s" in fpath:
            if ".field" in fpath: ch = "hmiB_field"
            elif ".inclination" in fpath: ch = "hmiB_incl"
            elif ".azimuth" in fpath: ch = "hmiB_azim"
        elif "hmi.m_720s" in fpath: ch = "hmiM"
        elif "hmi.ic_720s" in fpath: ch = "hmiIC"

        stacks.setdefault(ch, []).append(data)

    if not stacks:
        return False

    packed = {k: np.stack(v, axis=0) for k,v in stacks.items()}
    np.savez_compressed(out_npz, **packed)
    print(f"💾 Saved {out_npz} ({len(packed)} channels)")

    for k, v in packed.items():
        print(f"  {k:<10}: shape={v.shape}, mean={np.nanmean(v):.3f}, std={np.nanstd(v):.3f}")

    # --- Cleanup ---
    for fpath in fits_files:
        try:
            os.remove(fpath)
        except:
            pass
    print(f"🧹 Deleted {len(fits_files)} FITS from {save_dir}")
    return True

# ============================================================
# 📥 HMI DOWNLOADER
# ============================================================

def download_hmi_from_jsoc(start_time, end_time, save_dir, email):
    os.makedirs(save_dir, exist_ok=True)
    hmi_series = [
        ("hmi.B_720s", "field"),
        ("hmi.B_720s", "inclination"),
        ("hmi.B_720s", "azimuth"),
        ("hmi.M_720s", None),
        ("hmi.ic_720s", None),
    ]
    any_found = False
    for series, seg in hmi_series:
        try:
            attrs = [
                a.Time(start_time, end_time),
                a.jsoc.Series(series),
                a.Sample(12 * u.minute),
                a.jsoc.Notify(email),
            ]
            if seg: attrs.append(a.jsoc.Segment(seg))
            query = Fido.search(*attrs)
            if len(query) > 0:
                Fido.fetch(query, path=save_dir, max_conn=1, overwrite=False)
                any_found = True
        except Exception as e:
            print(f"⚠️ HMI {series} failed: {e}")
    return any_found

# ============================================================
# 🧭 MAIN LOOP — SMART DOWNLOAD (±1h → ±2h fallback)
# ============================================================

for flare_name, flare_start in FLARES:
    flare_dir = os.path.join(BASE_DIR, flare_name, "full_disk", "npz_hmi")
    os.makedirs(flare_dir, exist_ok=True)

    flare_start_dt = datetime.strptime(flare_start, "%Y-%m-%d %H:%M:%S")
    offsets = [-24, -18, -12, -6, 0, 6]

    print(f"\n===============================================")
    print(f"☀️ Processing {flare_name} ({flare_start})")
    print("===============================================")

    for h in offsets:
        t0 = flare_start_dt + timedelta(hours=h - 0.5)
        t1 = flare_start_dt + timedelta(hours=h + 0.5)
        tag = t0.strftime("%Y%m%dT%H%M")
        save_dir = os.path.join(flare_dir, tag)
        out_npz = os.path.join(flare_dir, f"{tag}.npz")
        os.makedirs(save_dir, exist_ok=True)

        if os.path.exists(out_npz):
            print(f"⏭️ Skipping {flare_name}/{tag} — already exists")
            continue

        print(f"\n🕓 Trying ±1h window: {t0} → {t1}")
        found = download_hmi_from_jsoc(
            t0.strftime("%Y-%m-%dT%H:%M:%S"),
            t1.strftime("%Y-%m-%dT%H:%M:%S"),
            save_dir, EMAIL
        )
        success = fits_to_npz_all_hmi(save_dir, out_npz)

        if not found or not success:
            print(f"⚠️ Retrying with ±2h window for {flare_name} offset {h}h...")
            t0_f = flare_start_dt + timedelta(hours=h - 1)
            t1_f = flare_start_dt + timedelta(hours=h + 1)
            found = download_hmi_from_jsoc(
                t0_f.strftime("%Y-%m-%dT%H:%M:%S"),
                t1_f.strftime("%Y-%m-%dT%H:%M:%S"),
                save_dir, EMAIL
            )
            success = fits_to_npz_all_hmi(save_dir, out_npz)
            if success:
                print(f"✅ Success on fallback (±2h) for {flare_name}/{tag}")
            else:
                print(f"🚫 No HMI data even with ±2h for {flare_name}/{tag}")

print("\n🎯 All HMI full-disk data processed successfully (±1h→±2h fallback, with cleanup).")
